1. Persiapan dan Instalasi Library

In [5]:
# Install library yang dibutuhkan
!pip install tensorflow pandas numpy scikit-learn sentence-transformers -q

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sentence_transformers import SentenceTransformer

# Mount Google Drive untuk mengakses dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


2. Memuat dan Memproses Data (Data Preprocessing)

In [6]:
# Tentukan path dataset
movies_path = '/content/drive/MyDrive/dataset sistem rekomendasi/movies.csv'
ratings_path = '/content/drive/MyDrive/dataset sistem rekomendasi/ratings.csv'

# Memuat dataset
print("Memuat dataset dari Google Drive...")
movies = pd.read_csv(movies_path)
ratings = pd.read_csv(ratings_path)

# Gabungkan data untuk mendapatkan metadata teks pada setiap rating
df = ratings.merge(movies, on='movieId')

# 1. Encoding User ID dan Item ID ke rentang kontinu
user_ids = df['userId'].astype('category').cat.codes.values
item_ids = df['movieId'].astype('category').cat.codes.values

df['user_encoded'] = user_ids
df['item_encoded'] = item_ids

num_users = df['user_encoded'].nunique()
num_items = df['item_encoded'].nunique()

print(f"Jumlah User: {num_users}, Jumlah Item: {num_items}")

# 2. Siapkan label (misal: memprediksi probabilitas interaksi positif / rating >= 3.5)
df['interaction'] = (df['rating'] >= 3.5).astype(int)

Memuat dataset dari Google Drive...
Jumlah User: 1819, Jumlah Item: 14018


3. Ekstraksi Fitur Teks dengan BERT (Feature Engineering)

In [7]:
# Inisialisasi pre-trained model BERT yang ringan dan cepat
print("Memuat model BERT (all-MiniLM-L6-v2)...")
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Membuat fitur teks dengan menggabungkan Judul dan Genre
# Memastikan penanganan string kosong atau null
movies['title'] = movies['title'].fillna('')
movies['genres'] = movies['genres'].fillna('').str.replace('|', ' ')
movies['text_feature'] = movies['title'] + " " + movies['genres']

# Mengekstrak embedding untuk setiap film unik
print("Mengekstrak embedding teks...")
item_text_embeddings = bert_model.encode(movies['text_feature'].tolist(), show_progress_bar=True)

# Buat dictionary/mapping dari item_encoded ke vektor BERT
item_id_to_encoded = dict(zip(movies['movieId'], movies['movieId'].astype('category').cat.codes))

# Buat matriks embedding teks yang sejajar dengan indeks item_encoded
embedding_dim_bert = item_text_embeddings.shape[1] # Umumnya 384 dimensi untuk MiniLM
bert_embedding_matrix = np.zeros((num_items, embedding_dim_bert))

for idx, row in movies.iterrows():
    encoded_id = item_id_to_encoded.get(row['movieId'])
    if pd.notna(encoded_id) and encoded_id < num_items:
        bert_embedding_matrix[int(encoded_id)] = item_text_embeddings[idx]

print(f"Bentuk Matriks BERT: {bert_embedding_matrix.shape}")

Memuat model BERT (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Mengekstrak embedding teks...


Batches:   0%|          | 0/1951 [00:00<?, ?it/s]

Bentuk Matriks BERT: (14018, 384)


4. Membangun Arsitektur Model BERT + NCF

In [8]:
# Hyperparameters
embedding_size = 50
hidden_units = [128, 64, 32]
dropout_rate = 0.2

# --- INPUT LAYERS ---
user_input = Input(shape=(1,), name='user_input')
item_input = Input(shape=(1,), name='item_input')
bert_input = Input(shape=(embedding_dim_bert,), name='bert_input')

# --- EMBEDDING LAYERS ---
# Collaborative Filtering Embeddings
user_embedding = Embedding(input_dim=num_users, output_dim=embedding_size, name='user_embedding')(user_input)
item_embedding = Embedding(input_dim=num_items, output_dim=embedding_size, name='item_embedding')(item_input)

# Flatten Embeddings
user_flatten = Flatten()(user_embedding)
item_flatten = Flatten()(item_embedding)

# Pemrosesan fitur BERT (Mereduksi dimensi teks agar lebih ringkas)
bert_dense = Dense(64, activation='relu', name='bert_dense')(bert_input)

# --- CONCATENATION ---
# Menggabungkan fitur CF dan fitur Teks
concat = Concatenate()([user_flatten, item_flatten, bert_dense])

# --- MULTI-LAYER PERCEPTRON (MLP) ---
mlp = Dense(hidden_units[0], activation='relu')(concat)
mlp = Dropout(dropout_rate)(mlp)
mlp = Dense(hidden_units[1], activation='relu')(mlp)
mlp = Dropout(dropout_rate)(mlp)
mlp = Dense(hidden_units[2], activation='relu')(mlp)

# --- OUTPUT LAYER ---
output = Dense(1, activation='sigmoid', name='prediction')(mlp)

# Compile Model
model = Model(inputs=[user_input, item_input, bert_input], outputs=output)
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 50)     │     90,950 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 50)     │    700,900 │ item_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_input          │ (None, 384)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 50)        │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 50)        │          0 │ item_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_dense (Dense)  │ (None, 64)        │     24,640 │ bert_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 164)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0],  │
│                     │                   │            │ bert_dense[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     21,120 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prediction (Dense)  │ (None, 1)         │         33 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 847,979 (3.23 MB)

 Trainable params: 847,979 (3.23 MB)

 Non-trainable params: 0 (0.00 B)

5. Persiapan Data Latih dan Pelatihan Model (Training)

In [9]:
# Siapkan input arrays untuk training
X_users = df['user_encoded'].values
X_items = df['item_encoded'].values

# Petakan vektor BERT ke setiap baris transaksi berdasarkan item_encoded
X_bert = np.array([bert_embedding_matrix[i] for i in X_items])
y = df['interaction'].values

# Train-Test Split (80% Train, 20% Test)
print("Memisahkan data latih dan data uji...")
X_u_train, X_u_test, X_i_train, X_i_test, X_b_train, X_b_test, y_train, y_test = train_test_split(
    X_users, X_items, X_bert, y, test_size=0.2, random_state=42
)

# Proses Training
print("Memulai proses training...")
history = model.fit(
    x=[X_u_train, X_i_train, X_b_train],
    y=y_train,
    batch_size=256,
    epochs=10,
    validation_split=0.1
)

Memisahkan data latih dan data uji...
Memulai proses training...
Epoch 1/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - accuracy: 0.7130 - loss: 0.5563 - val_accuracy: 0.7369 - val_loss: 0.5282
Epoch 2/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.7550 - loss: 0.5025 - val_accuracy: 0.7374 - val_loss: 0.5230
Epoch 3/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.7682 - loss: 0.4787 - val_accuracy: 0.7402 - val_loss: 0.5284
Epoch 4/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.7806 - loss: 0.4584 - val_accuracy: 0.7377 - val_loss: 0.5371
Epoch 5/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.7932 - loss: 0.4370 - val_accuracy: 0.7340 - val_loss: 0.5479
Epoch 6/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.8076 - loss: 0.4132 - val_accuracy: 0.7333 - val_loss: 0.5790
Epoch 7/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.8222 - loss: 0.3868 - val_accuracy: 0.7275 - val_loss: 0.6181
Epoch 8/10
723/723 ━━━━━━━━

6. Evaluasi dan Fungsi Rekomendasi

In [13]:
# Evaluasi pada data uji (Test Set)
print("\nMengevaluasi model pada data uji...")
loss, accuracy = model.evaluate([X_u_test, X_i_test, X_b_test], y_test)
print(f'Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}')

# Fungsi untuk memberikan Top-N Rekomendasi
def recommend_movies(user_id_encoded, top_n=5):
    # Validasi apakah user valid
    if user_id_encoded >= num_users or user_id_encoded < 0:
        print(f"Error: user_id_encoded harus antara 0 dan {num_users-1}")
        return

    # Buat array item untuk semua film yang ada
    all_items = np.arange(num_items)
    user_array = np.full(num_items, user_id_encoded)

    # Ambil fitur BERT untuk semua item
    all_bert_features = bert_embedding_matrix

    # Prediksi skor probabilitas
    predictions = model.predict([user_array, all_items, all_bert_features], batch_size=512, verbose=0).flatten()

    # Ambil indeks item dengan skor tertinggi
    top_indices = predictions.argsort()[-top_n:][::-1]

    print(f"\nRekomendasi Top-{top_n} untuk User Index {user_id_encoded}:")
    for idx in top_indices:
        # Kembalikan ke ID asli dan cari nama filmnya
        try:
            original_movie_id = list(item_id_to_encoded.keys())[list(item_id_to_encoded.values()).index(idx)]
            movie_title = movies[movies['movieId'] == original_movie_id]['title'].values[0]
            score = predictions[idx]
            print(f"- {movie_title} (Skor Prediksi: {score:.4f})")
        except ValueError:
            continue

# Uji sistem rekomendasi untuk user index 0
recommend_movies(user_id_encoded=0, top_n=5)


Mengevaluasi model pada data uji...
1606/1606 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7195 - loss: 0.7249
Test Loss: 0.7249, Test Accuracy: 0.7195

Rekomendasi Top-5 untuk User Index 0:
- Radio Days (1987) (Skor Prediksi: 1.0000)
- Lotto Land (1995) (Skor Prediksi: 1.0000)
- Sorority House Massacre II (1990) (Skor Prediksi: 1.0000)
- Rock Star (2001) (Skor Prediksi: 1.0000)
- I'm All Right Jack (1959) (Skor Prediksi: 1.0000)
